# CS383: Data Science and Machine Learning
## Lecture 9 Exercises — Evaluation Metrics for Classification

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File → Save and Export Notebook As → Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 9 Exercise** as a Jupyter Notebook
(.ipynb) file.

### Setup — NYC restaurant inspections

A different real dataset than the lecture notebook, same toolkit. Target: **is this inspection's grade a
"C"?** — a genuinely rare outcome (most restaurants get an A), so this stays a real imbalanced problem,
not a toy one.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score,
    f1_score, classification_report, roc_curve, roc_auc_score,
)

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "grade", "boro", "cuisine_description"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

    top_cuisines = inspections_df["cuisine_description"].value_counts().nlargest(10).index
    inspections_df["cuisine_top"] = np.where(
        inspections_df["cuisine_description"].isin(top_cuisines), inspections_df["cuisine_description"], "Other"
    )
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 6000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_top": rng.choice(cuisines_clean, size=n),
        "is_critical": rng.integers(0, 2, size=n),
    })
    inspections_df["grade"] = rng.choice(["A", "B", "C"], size=n, p=[0.62, 0.25, 0.13])
    live = False

# Target: is this a "C" grade? NOTE: we deliberately do NOT use `score` as a feature here -- Lecture 7
# showed that grade is computed by thresholding score directly, so using score to predict grade would be
# the exact same target-leakage problem all over again. `boro`, `cuisine_top`, and `is_critical` are
# genuinely independent information, known without needing the score.
inspections_df["is_grade_c"] = (inspections_df["grade"] == "C").astype(int)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} records")
print(f"'Grade C' rate: {inspections_df['is_grade_c'].mean():.1%}")

---

## Exercise 1 — Put It Together

**Scenario:** predict whether a restaurant inspection earned a "C" grade, using `boro`, `cuisine_top`, and
`is_critical` — genuinely independent features, not the score itself. Same evaluation toolkit as the
lecture: baseline, confusion matrix, precision/recall/F1, threshold tuning, ROC/AUC.

### Step 1 — The baseline you have to beat

In [ ]:
majority_baseline = max(inspections_df["is_grade_c"].mean(), 1 - inspections_df["is_grade_c"].__________())
print(f"Majority-class baseline accuracy: {majority_baseline:.1%}")

### Step 2 — Split, preprocess, and fit

In [ ]:
X = inspections_df[["boro", "cuisine_top", "is_critical"]]
y = inspections_df["is_grade_c"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ("cat", __________(handle_unknown="ignore"), ["boro", "cuisine_top"]),
], remainder="passthrough")

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model.__________(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Training rows:", len(X_train), "| Test rows:", len(X_test))

### Step 3 — Confusion matrix and classification report

In [ ]:
cm = __________(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["not C", "C"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion Matrix — Grade C Classifier")
plt.show()

print(f"Accuracy: {(y_pred == y_test).mean():.3f}  (baseline: {majority_baseline:.3f})")
print()
print(__________(y_test, y_pred, target_names=["not C", "C"]))

### Step 4 — Threshold tuning

`is_grade_c` is rarer than the lecture's `is_late` (about 5% here), so the model's predicted
probabilities rarely climb anywhere near 0.5 even when it's genuinely onto something -- watch what
happens to precision and recall as the threshold comes down to match that.

In [ ]:
for threshold in [0.15, 0.10, 0.08]:
    preds_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, preds_t, zero_division=0)
    r = __________(y_test, preds_t)
    print(f"threshold={threshold:.2f}: precision={p:.3f}, recall={r:.3f}, flagged={preds_t.sum()}")

### Step 5 — ROC curve and AUC

In [ ]:
fpr, tpr, _ = __________(y_test, y_proba)
auc = __________(y_test, y_proba)

plt.plot(fpr, tpr, color="#2B6CB0", linewidth=2, label=f"This model (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], color="#9AA5B1", linestyle="--", label="Random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Grade C Classifier")
plt.legend()
plt.show()

### Step 6 — Explain it back

In 2-3 sentences: at the default 0.5 threshold, is this model more useful for *finding* C-grade
restaurants (recall) or for *trusting a flag* when it fires (precision)? What does the ROC/AUC tell you
that the 0.5-threshold numbers alone don't?

**Your explanation:**

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. A model gets 92% accuracy on a problem where 91% of cases are the negative class. Is that impressive?
   What single number would you check next before deciding?
2. Give a real-world example (not from this notebook) where you'd care more about recall than precision,
   and one where you'd care more about precision than recall. Explain the cost of each kind of error in
   your examples.
3. If you lower the classification threshold from 0.5 to 0.2, what happens to recall? What happens to
   precision? Is either change guaranteed, or just typical?
4. Why can AUC stay well above 0.5 even when accuracy barely beats the majority-class baseline? What is
   AUC measuring that accuracy at a single threshold isn't?
5. What question do you still have about classification metrics heading into Week 10?

**Your responses:**

1.
2.
3.
4.
5. 

## Optional Challenge

Apply the same toolkit to NYC 311 again, but change how "late" is defined, and see how the metrics react
to a *less* imbalanced problem.

### Setup — NYC 311, with a looser "late" cutoff

In [ ]:
try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live311 = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.05
    resolution_hours = rng.gamma(shape=1.5, scale=22.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")
    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live311 = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

# A looser definition: "late" now means more than 24 hours, not 72 -- a much less rare event.
complaints_df["is_late_24h"] = (complaints_df["resolution_time_hours"] > 24).astype(int)

print(f"{'Shared snapshot' if live311 else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
print(f"'Late' (>24h) rate: {complaints_df['is_late_24h'].mean():.1%}  (compare to >72h in the lecture)")

### Step 1 — Split, preprocess, and fit

Use `hour_filed`, `borough`, `complaint_grouped` to predict `is_late_24h`, same `Pipeline` pattern as the
lecture.

In [ ]:
# Your code here


### Step 2 — Confusion matrix, classification report, and AUC

Compute all of it for this looser target.

In [ ]:
# Your code here


### Step 3 — Reflect

Compare this model's recall at the default 0.5 threshold to the lecture's `>72h` model. Which target was
easier for the classifier to get right at 0.5, and does that match what you'd expect from a *less*
imbalanced problem? What happened to AUC?

**Your answer:**

### Big idea
> How imbalanced a target is isn't a detail — it changes which metrics are trustworthy and which
> threshold is reasonable. Always check the class balance before picking a metric, not after being
> surprised by one.